In [6]:
import re
import numpy as np
import pandas as pd

TEST_FILE = "Private Ryan Script.txt"

MANUAL_GENRE_VECTORS = {
    "Horror": [
        "blood", "scream", "killer", "dark", "death", "fear", "ghost", "knife", "night", "evil",
        "dead", "body", "monster", "shadow", "haunted", "attack", "hide", "danger", "panic", "cry",
        "terrified", "grave", "curse", "demon", "victim", "murder", "skull", "forest", "shock", "creature"
    ],
    "Crime": [
        "police", "money", "gun", "car", "bank", "street", "boss", "deal", "stole", "detective",
        "robbery", "murder", "case", "criminal", "evidence", "cop", "mafia", "hit", "jail", "drug",
        "suspect", "thief", "lawyer", "court", "gang", "escape", "witness", "track", "investigate", "bullet"
    ],
    "War": [
        "army", "battle", "soldier", "enemy", "captain", "general", "attack", "bomb", "fight", "weapon",
        "mission", "troop", "tank", "warfare", "gunfire", "uniform", "commander", "base", "front", "march",
        "victory", "defeat", "radio", "orders", "camp", "battlefield", "explosion", "rifle", "navy", "aircraft"
    ]
}

STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from", "had", "has", "have",
    "he", "her", "hers", "him", "his", "i", "if", "in", "into", "is", "it", "its", "me", "my",
    "of", "on", "or", "our", "she", "that", "the", "their", "them", "they", "this", "to", "was",
    "we", "were", "with", "you", "your"
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [word for word in tokens if word not in STOP_WORDS and len(word) > 1]

def read_text_file(file_name):
    with open(file_name, "r", encoding="utf-8") as f:
        return f.read()

def count_matches(tokens, genre_words):
    counts = pd.Series(tokens).value_counts()
    return sum(counts.get(word, 0) for word in genre_words)

def cosine_similarity(vec1, vec2):
    denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denom == 0:
        return 0.0
    return float(np.dot(vec1, vec2) / denom)

def main():
    script_text = read_text_file(TEST_FILE)
    tokens = clean_text(script_text)

    vocabulary = []
    for words in MANUAL_GENRE_VECTORS.values():
        for word in words:
            if word not in vocabulary:
                vocabulary.append(word)

    script_vector = np.array([tokens.count(word) for word in vocabulary], dtype=float)

    scores = {}
    for genre, words in MANUAL_GENRE_VECTORS.items():
        genre_vector = np.array([1 if word in words else 0 for word in vocabulary], dtype=float)
        scores[genre] = cosine_similarity(script_vector, genre_vector)

    print("Similarity scores:")
    for genre, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        print(genre, round(score, 4))

    print("\nRaw matching word counts:")
    for genre, words in MANUAL_GENRE_VECTORS.items():
        print(genre, count_matches(tokens, words))

    predicted_genre = max(scores, key=scores.get)
    print("\nPredicted genre:", predicted_genre)

main()

Similarity scores:
War 0.326
Horror 0.1373
Crime 0.129

Raw matching word counts:
Horror 99
Crime 93
War 235

Predicted genre: War
